# Path-Selection Policy Sketch

A policy algorithm that selects which coder path to compile and run next,
driven by the DSL algebra and an energy signal. No real tensors required.

**Three rules** (Lang's algorithm translated into the Lambert algebra):
- **Push** — extend the current path by one leg (energy improving)
- **Replace** — fold the path via `coder.compile` if it reduces (idempotence)
- **Pop** — backtrack one leg (energy worsening)

Candidate paths at each Push are aggregated via logsumexp (⊕_T).
At T→0 this is argmin energy (greedy). At T>0 it maintains a soft mixture.

**To use real legs**: replace each mock lambda in Cell 2 with
`tensor.Join(...)` or `tensor.Residuate(...)`. Nothing else changes.

In [1]:
import sys, dataclasses
import numpy as np
from scipy.special import logsumexp
from dataclasses import dataclass
from typing import Optional

sys.path.insert(0, '..')
from lattice.coder import PathCoder, LEG_TYPE, _SAME_PAIR, _MIXED_PAIR

In [2]:
# Mock legs — return tokens instead of tensors.
# Swap each lambda for a real Join/Residuate call to go to production.
legs = [
    lambda x, y, t: ("realize",   x, y),   # se : C → E
    lambda x, y, t: ("propagate", x, y),   # sd : E → C
    lambda x, y, t: ("abstract",  x, y),   # pe : E → C
    lambda x, y, t: ("support",   x, y),   # pd : C → E
]
coder = PathCoder(legs, fold_empirical=False)

# Confirm coder.compile already handles folding correctly
print("parse 'se sd'             :", coder.parse("se sd").steps)
print("parse 'sd pd sd pd'       :", coder.parse("sd pd sd pd").steps)  # folds to sd pd
print("compile fold_empirical    :", coder.compile("se sd se sd", fold_empirical=True).steps)

parse 'se sd'             : (Step(name='realize', swap=False, same=False, trans=False), Step(name='propagate', swap=False, same=False, trans=False))
parse 'sd pd sd pd'       : (Step(name='propagate', swap=False, same=False, trans=False), Step(name='support', swap=False, same=False, trans=False))
compile fold_empirical    : (Step(name='realize', swap=False, same=False, trans=False), Step(name='propagate', swap=False, same=False, trans=False))


In [3]:
@dataclass
class PolicyState:
    """
    State of the path-selection policy at one point in time.

    spec          : current path as a spec string — single source of truth.
                    coder.parse(spec) is the authority on its meaning.
    initial_space : starting space ("C" or "E"), never mutated.
                    Used to determine valid first legs and to recover space
                    after a pop back to empty.
    energy        : energy reading for the current spec.
    prev_energy   : energy from the previous step — drives push vs. pop.
    score         : log-weight of this path under the policy semiring.
    step_count    : number of push steps taken (replace/pop are free).
    """
    spec:          str
    initial_space: str
    energy:        float
    prev_energy:   float
    score:         float = 0.0
    step_count:    int   = 0

    @property
    def current_space(self) -> str:
        """Output space of the last leg, or initial_space if spec is empty."""
        tokens = self.spec.split()
        if not tokens:
            return self.initial_space
        return LEG_TYPE[tokens[-1]][1]

    def __repr__(self):
        return (f"PolicyState({self.spec!r:40s} "
                f"space={self.current_space} E={self.energy:.3f})")


def initial_state(start_space: str = "C", initial_energy: float = 1.0) -> PolicyState:
    return PolicyState(
        spec="",
        initial_space=start_space,
        energy=initial_energy,
        prev_energy=float("inf"),
    )

print(initial_state("C"))
print("current_space of empty C state:", initial_state("C").current_space)
print("current_space of 'realize'    :", dataclasses.replace(initial_state("C"), spec="realize").current_space)

PolicyState(''                                       space=C E=1.000)
current_space of empty C state: C
current_space of 'realize'    : E


In [4]:
def push(state: PolicyState) -> list:
    """
    Enumerate all valid one-leg extensions from the current output space.

    Uses LEG_TYPE to find legs whose input matches current_space.
    From C: {realize, support} — branching factor 2.
    From E: {propagate, abstract} — branching factor 2.

    The branching factor is always exactly 2 — a consequence of the two
    adjoint pairs (sd⊣pd, pe⊣se): each space has exactly two legs leaving it,
    one from each pair.
    """
    candidates = [
        name for name, (inp, out) in LEG_TYPE.items()
        if inp == state.current_space
    ]
    return [
        dataclasses.replace(
            state,
            spec=f"{state.spec} {name}".strip(),
            step_count=state.step_count + 1,
        )
        for name in candidates
    ]


def try_replace(state: PolicyState, coder: PathCoder,
                fold_empirical: bool = False) -> Optional[PolicyState]:
    """
    Delegate to coder.compile to fold the current spec.

    coder.compile already calls _fold(_SAME_PAIR) via parse(), and optionally
    _fold(_MIXED_PAIR) when fold_empirical=True. If the compiled path has fewer
    steps than the input tokens, a fold happened — return the reduced state.

    This is the entire Replace rule. No manual pair-checking needed.
    """
    if not state.spec:
        return None
    path = coder.compile(state.spec, fold_empirical=fold_empirical)
    folded = " ".join(s.name for s in path.steps)
    if folded == state.spec:
        return None
    return dataclasses.replace(state, spec=folded)


def pop(state: PolicyState, n: int = 1) -> Optional[PolicyState]:
    """
    Remove the last n legs. Returns None if path has fewer than n legs.
    """
    tokens = state.spec.split()
    if len(tokens) < n:
        return None
    return dataclasses.replace(state, spec=" ".join(tokens[:-n]))


# Unit tests
s = initial_state("C")
print("Push from empty C:", [c.spec for c in push(s)])

s2 = dataclasses.replace(s, spec="realize")
print("Push from E      :", [c.spec for c in push(s2)])

# Replace: single same-pair occurrence — NOT folded (no repeat)
s3 = dataclasses.replace(s, spec="support propagate")
print("Single pd sd (no fold):", try_replace(s3, coder))

# Replace: repeated same-pair — coder.compile folds it
s4 = dataclasses.replace(s, spec="support propagate support propagate")
print("Repeated pd sd (fold) :", try_replace(s4, coder))

# Replace: repeated mixed-pair — only folds with fold_empirical=True
s5 = dataclasses.replace(s, spec="realize propagate realize propagate")
print("Repeated Attend no fold:", try_replace(s5, coder, fold_empirical=False))
print("Repeated Attend fold   :", try_replace(s5, coder, fold_empirical=True))

# Pop
print("Pop s4 by 1:", pop(s4, 1).spec)

Push from empty C: ['realize', 'support']
Push from E      : ['realize propagate', 'realize abstract']
Single pd sd (no fold): None
Repeated pd sd (fold) : PolicyState('support propagate'                      space=C E=1.000)
Repeated Attend no fold: None
Repeated Attend fold   : PolicyState('realize propagate'                      space=C E=1.000)
Pop s4 by 1: support propagate support


In [5]:
# Mock energy oracle.
# Assigns scalar energies to path spec strings. Lower = better fit.
#
# Two low-energy basins — one per adjoint pair:
#   sd⊣pd family: "support propagate" (Correct), "propagate support"
#   pe⊣se family: "realize abstract", "abstract realize"
#
# Three-step extensions of those basins go lower still.
# Repeated paths (which coder.compile would fold) get higher energy —
# the Replace rule fires before they persist.

MOCK_ENERGY = {
    # single legs — underspecified
    "realize":   0.90,
    "propagate": 0.90,
    "abstract":  0.90,
    "support":   0.90,

    # two-step same-pair roundtrips
    "support propagate":   0.28,   # pd sd = Correct
    "propagate support":   0.30,   # sd pd
    "realize abstract":    0.28,   # se pe
    "abstract realize":    0.30,   # pe se

    # two-step mixed pairs (Attend / Recall)
    "realize propagate":   0.40,   # se sd = Attend
    "abstract support":    0.42,   # pe pd = Recall
    "support abstract":    0.55,
    "propagate realize":   0.50,

    # three-step: extend a low-energy two-step
    "support propagate realize":    0.20,
    "support propagate abstract":   0.22,
    "realize abstract realize":     0.18,
    "realize abstract support":     0.21,
    "abstract realize abstract":    0.19,
    "abstract realize propagate":   0.22,
    "propagate support propagate":  0.20,
    "propagate support realize":    0.23,

    # repeated paths — high energy (coder.compile would fold these)
    "support propagate support propagate": 0.45,
    "realize propagate realize propagate": 0.48,
}

def mock_energy(spec: str) -> float:
    return MOCK_ENERGY.get(spec.strip(), 0.60)

# Display the landscape
print(f"{'spec':<44s}  energy  basin")
print("-" * 60)
for spec in sorted(MOCK_ENERGY, key=lambda s: (len(s.split()), MOCK_ENERGY[s])):
    basin = ("sd⊣pd" if any(t in spec for t in ["support","propagate"]) and
             "realize" not in spec and "abstract" not in spec
             else "pe⊣se" if any(t in spec for t in ["realize","abstract"]) and
             "support" not in spec and "propagate" not in spec
             else "mixed")
    print(f"  {spec:<42s}  {MOCK_ENERGY[spec]:.2f}    {basin}")

spec                                          energy  basin
------------------------------------------------------------
  realize                                     0.90    pe⊣se
  propagate                                   0.90    sd⊣pd
  abstract                                    0.90    pe⊣se
  support                                     0.90    sd⊣pd
  support propagate                           0.28    sd⊣pd
  realize abstract                            0.28    pe⊣se
  propagate support                           0.30    sd⊣pd
  abstract realize                            0.30    pe⊣se
  realize propagate                           0.40    mixed
  abstract support                            0.42    mixed
  propagate realize                           0.50    mixed
  support abstract                            0.55    mixed
  realize abstract realize                    0.18    pe⊣se
  abstract realize abstract                   0.19    pe⊣se
  support propagate realize            

In [6]:
def aggregate(candidates: list, temperature: float) -> tuple:
    """
    Score all candidate paths and aggregate via logsumexp (⊕_T).

    Score = -energy / T   (Boltzmann log-weight)

    At T→0: collapses to argmin energy (greedy/Viterbi).
    At T>0: soft mixture over all candidates.

    Returns the highest-scoring candidate and the full normalised log-weight list.

    Design flag D: hard argmax is used here. A full soft-blend would execute
    all candidate paths simultaneously and weight their outputs — requires
    vectorised multi-path execution.

    Design flag H: in production T should come from FixpointIterator._update_temp()
    (core/fixpoint.py:97-115) rather than being a fixed caller argument.
    """
    raw = [
        (-mock_energy(c.spec) / temperature if temperature > 1e-9
         else -mock_energy(c.spec) * 1e6)
        for c in candidates
    ]
    log_Z = logsumexp(raw)
    log_w = [r - log_Z for r in raw]
    best_i = int(np.argmax(raw))
    best = dataclasses.replace(
        candidates[best_i],
        energy=mock_energy(candidates[best_i].spec),
        score=raw[best_i],
    )
    return best, log_w


# Demo: aggregation from empty C state
s0 = initial_state("C", 1.0)
cands = push(s0)
print(f"Candidates: {[c.spec for c in cands]}\n")
print(f"{'T':<6}  probs                   dominant")
for T in [0.02, 0.1, 0.5, 1.0, 2.0, 5.0]:
    best, lw = aggregate(cands, T)
    probs = np.exp(lw)
    print(f"T={T:<4.2f}  {[round(p,3) for p in probs]}   {best.spec!r}")

Candidates: ['realize', 'support']

T       probs                   dominant
T=0.02  [np.float64(0.5), np.float64(0.5)]   'realize'
T=0.10  [np.float64(0.5), np.float64(0.5)]   'realize'
T=0.50  [np.float64(0.5), np.float64(0.5)]   'realize'
T=1.00  [np.float64(0.5), np.float64(0.5)]   'realize'
T=2.00  [np.float64(0.5), np.float64(0.5)]   'realize'
T=5.00  [np.float64(0.5), np.float64(0.5)]   'realize'


In [7]:

import sys
sys.path.insert(0, '..')
from core.fixpoint import FixpointIterator

def make_policy_step(initial_space, fold_empirical=False, max_pop_attempts=3):
    """
    Return a step function (energy_arr, temp) -> new_energy_arr
    suitable for FixpointIterator.

    State = 1-element float array [mock_energy(current_spec)].
    FixpointIterator drives the convergence check and temperature annealing
    via _update_temp — no manual energy/delta tracking needed here.

    The spec is carried in a mutable closure cell; history is recorded there.
    Returns new_state only (no aux) so FixpointIterator.default_energy uses
    only the dynamic error term ||new - old||².
    """
    cell = {
        "spec":             "",
        "space":            initial_space,
        "consecutive_pops": 0,
        "history":          [],
    }

    def step(energy_arr, temp):
        spec = cell["spec"]

        # Replace (free, eager) — delegate entirely to coder.compile
        if spec:
            path   = coder.compile(spec, fold_empirical=fold_empirical)
            folded = " ".join(s.name for s in path.steps)
            if folded != spec:
                cell["spec"] = folded
                spec = folded

        current_space  = LEG_TYPE[spec.split()[-1]][1] if spec else cell["space"]
        current_energy = energy_arr[0]

        # Push: enumerate valid extensions, aggregate via -energy/temp (logsumexp)
        candidate_names    = [n for n, (inp, _) in LEG_TYPE.items() if inp == current_space]
        candidate_specs    = [f"{spec} {n}".strip() for n in candidate_names]
        candidate_energies = [mock_energy(s) for s in candidate_specs]

        raw   = ([-e / temp for e in candidate_energies] if temp > 1e-9
                 else [-e * 1e6 for e in candidate_energies])
        best_i      = int(np.argmax(raw))
        best_spec   = candidate_specs[best_i]
        best_energy = candidate_energies[best_i]

        if best_energy <= current_energy:
            # Push accepted — energy still falling
            cell["spec"]             = best_spec
            cell["consecutive_pops"] = 0
        else:
            # Pop — energy would rise; backtrack one leg
            cell["consecutive_pops"] += 1
            tokens = spec.split()
            if tokens and cell["consecutive_pops"] <= max_pop_attempts:
                cell["spec"] = " ".join(tokens[:-1])
            # If max pops exceeded or nothing to pop, stay put.
            # FixpointIterator sees ||new-old||² = 0 → converge.

        new_energy = mock_energy(cell["spec"])
        cell["history"].append((cell["spec"], new_energy, temp))
        # Return new_state only — aux=None so default_energy uses dynamic error only
        return np.array([new_energy])

    step.cell = cell   # expose for inspection after run
    return step

print("make_policy_step defined.")


make_policy_step defined.


In [8]:

SEP = "=" * 65

def run_fp(initial_space="C", init_temp=1.0, fold_empirical=False, verbose=True):
    """Run make_policy_step inside FixpointIterator. Returns (fp, step_fn)."""
    step_fn = make_policy_step(initial_space, fold_empirical=fold_empirical)
    fp = FixpointIterator(
        f      = step_fn,
        state0 = np.array([1.0]),
        eps    = 1e-4,    # tighter threshold so the iterator walks the landscape
        max_iters = 20,
        temp   = init_temp,
    )
    fp.run(verbose=verbose)
    return fp, step_fn

# ── Exp 1: Standard run from C, T=1.0 (anneals automatically) ───────────────
print(SEP)
print("Exp 1: standard run from C, init_temp=1.0")
print(SEP)
fp1, s1 = run_fp("C", init_temp=1.0)
print("\nHistory:")
for spec, e, t in s1.cell["history"]:
    print(f"  {(spec or '(empty)'):<44s}  E={e:.3f}  T={t:.4f}")
print(f"Final: {s1.cell['spec']!r}  E={s1.cell['history'][-1][1]:.3f}")

# ── Exp 2: High vs low init_temp ─────────────────────────────────────────────
print(f"\n{SEP}")
print("Exp 2: init_temp=0.05 (greedy) vs init_temp=2.0 (exploratory)")
print(SEP)
for T in [0.05, 2.0]:
    fp, s = run_fp("C", init_temp=T, verbose=False)
    final_spec = s.cell["spec"]
    final_e    = s.cell["history"][-1][1]
    print(f"T={T:.2f}  final={final_spec!r}  E={final_e:.3f}  iters={fp._iter}")

# ── Exp 3: Starting from E ────────────────────────────────────────────────────
print(f"\n{SEP}")
print("Exp 3: starting from E (entity-space query)")
print(SEP)
fp3, s3 = run_fp("E", init_temp=1.0)
print(f"\nFinal: {s3.cell['spec']!r}  E={s3.cell['history'][-1][1]:.3f}")

# ── Exp 4: Replace demo ────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("Exp 4: Replace demo — coder.compile folds a repeated same-pair")
print(SEP)
s_rep = dataclasses.replace(
    initial_state("C"), spec="support propagate support propagate", energy=0.45, prev_energy=0.28)
result = try_replace(s_rep, coder)
print(f"Input : {s_rep.spec!r}")
print("Output:", repr(result.spec) if result else "(no fold)")

# fold_empirical test uses a fresh coder instance to avoid cache collision
# (PathCoder caches by spec string; a cached no-fold result blocks the fold_empirical path)
coder_fresh = PathCoder(legs, fold_empirical=False)
s_att = dataclasses.replace(
    initial_state("C"), spec="realize propagate realize propagate", energy=0.48, prev_energy=0.40)
r_false = try_replace(s_att, coder_fresh, fold_empirical=False)
r_true  = try_replace(s_att, coder_fresh, fold_empirical=True)
print(f"\nInput (Attend x2, fold_empirical=False): {s_att.spec!r}")
print("Output:", repr(r_false.spec) if r_false else "(no fold)")
print(f"Input (Attend x2, fold_empirical=True) : {s_att.spec!r}")
print("Output:", repr(r_true.spec) if r_true else "(no fold)")

# ── Exp 5: Semiring aggregation — E-space candidates have different energies ──
print(f"\n{SEP}")
print("Exp 5: semiring aggregation from E-space (after 'realize')")
print("  'realize propagate' E=0.40  vs  'realize abstract' E=0.28")
print(SEP)
s_e    = dataclasses.replace(initial_state("C"), spec="realize")
cands_e = push(s_e)
print(f"Candidates: {[c.spec for c in cands_e]}")
print(f"\n{'T':<6}  probs                          dominant")
for T in [0.02, 0.1, 0.5, 1.0, 2.0, 5.0]:
    best, lw = aggregate(cands_e, T)
    probs = np.exp(lw)
    print(f"T={T:<4.2f}  {str([round(p, 3) for p in probs]):<30s}  {best.spec!r}")


Exp 1: standard run from C, init_temp=1.0
  iter   1  energy=0.010000  temp=1.000000
  iter   2  energy=0.384400  temp=3.648426
  iter   3  energy=0.010000  temp=0.007856
  iter   4  energy=0.010000  temp=0.005832
  iter   5  energy=0.010000  temp=0.007856
  iter   6  energy=0.010000  temp=0.005832
  iter   7  energy=0.010000  temp=0.007856
  iter   8  energy=0.010000  temp=0.005832
  iter   9  energy=0.010000  temp=0.007856
  iter  10  energy=0.010000  temp=0.005832
  iter  11  energy=0.010000  temp=0.007856
  iter  12  energy=0.010000  temp=0.005832
  iter  13  energy=0.010000  temp=0.007856
  iter  14  energy=0.010000  temp=0.005832
  iter  15  energy=0.010000  temp=0.007856
  iter  16  energy=0.010000  temp=0.005832
  iter  17  energy=0.010000  temp=0.007856
  iter  18  energy=0.010000  temp=0.005832
  iter  19  energy=0.010000  temp=0.007856
  iter  20  energy=0.010000  temp=0.005832

History:
  realize                                       E=0.900  T=1.0000
  realize abstract    